# Installing Dependencies


In [1]:
# use this cell to install packages if needed
#!pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu116 --upgrade

In [2]:
# use this cell to install packages if needed
#!pip install transformers --upgrade

In [3]:
#!pip install tqdm

In [4]:
#!pip install tensorboard

# Setup

In [5]:
import json
import os
import timeit
import collections
import time
from pprint import pprint
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, squad_convert_examples_to_features
from transformers.data.processors.squad import SquadV2Processor,SquadResult
from transformers.data.metrics.squad_metrics import (
    compute_predictions_log_probs,
    compute_predictions_logits,
    squad_evaluate,
)

In [6]:
DO_LOWER_CASE = True
NBEST_SIZE = 20
DOC_STRIDE = 128
MAX_SEQ_LENGTH = 384
MAX_QUERY_LENGTH = 64
MAX_ANSWER_LENGTH = 30

# مسیر پوشه داده‌ها روی محیط ابری شما
DATA_DIR = './Data/Squad'
PREDICT_FILE = 'dev-v2.0.json'

BERT_MODEL_TYPE = 'bert'
BERT_MODEL_HF_PATH = "twmkn9/bert-base-uncased-squad2"
BERT_OUTPUT_DIR = "models/bert/twmkn9_bert-base-uncased-squad2"

DISTILBERT_MODEL_TYPE = 'distilbert'
DISTILBERT_MODEL_HF_PATH = 'twmkn9/distilbert-base-uncased-squad2'
DISTILBERT_OUTPUT_DIR = 'models/distilbert/twmkn9_distilbert-base-uncased-squad2'

print("Configurations are set successfully!")

Configurations are set successfully!


# Q&A Challenges

Measuring the success of Q&A systems is complicated. When asked a question like "Why the sky is Blue?", there are several potential right answers. For instance, one could refer to "Rayleigh Scattering" or another answer could be:
```
The Earth's atmosphere scatters short-wavelength light more efficiently than that of longer wavelengths. Because its wavelengths are shorter, blue light is more strongly scattered than the longer-wavelength lights, red or green. Hence the result that when looking at the sky away from the direct incident sunlight, the human eye perceives the sky to be blue.
```

Both options are correct and referred in Wikipedia Artile 'Diffuse Sky Radiation'


Most Q&A systems rely on a corpus of information that is initially indexed by an information retrieval system. Then, snippets of text are extracted where the Q&A model scores the most likely sentences to answer a given query.

However, the same snippet of text that answers the blue sky question, may not be able to answer a similar query like "Could the Sky ever be green?"

This is the gray area for measuring performance of Q&A models. How should we judge a model's success when there are multiple correct answers, even more incorrect answers, and, potentially no answers available to it at all?

# SQuAD Dataset

```
Stanford Question Answering Dataset (SQuAD) is a reading comprehension dataset, consisting of questions posed by crowdworkers on a set of Wikipedia articles, where the answer to every question is a segment of text, or span, from the corresponding reading passage, or the question might be unanswerable.
```

```
SQuAD2.0 combines the 100,000 questions in SQuAD1.1 with over 50,000 unanswerable questions written adversarially by crowdworkers to look similar to answerable ones. To do well on SQuAD2.0, systems must not only answer questions when possible, but also determine when no answer is supported by the paragraph and abstain from answering.
```

We target the SQuAD 2.0 as it represents a scenario that is closer to the real world: **It includes additional questions that cannot be answered by the accompanying passage**

**Download the Squad dev set for model evaluation**

In [7]:
#!wget -P data/squad/ https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v2.0.json

We will make extensive use of [Hugging Face](https://huggingface.co/) and [Pytorch](https://pytorch.org/) throughout this code, they provide several implementations for loading datasets and models

## Loading the DEV set using Hugging Face data processors

We will make use of [Processors](https://huggingface.co/transformers/main_classes/processors.html) to facilitate basic processing tasks with some canonical NLP datasets. The processors can be used for loading datasets and converting their examples to features for direct use in the model. More specifically, we will be using the [SQuAD processors](https://huggingface.co/transformers/main_classes/processors.html#squad)

In [8]:
def to_list(tensor):
    return tensor.detach().cpu().tolist()

In [9]:
def load_and_cache_examples(model_name_or_path, 
                            data_dir= DATA_DIR, 
                            predict_file=PREDICT_FILE, 
                            max_seq_length=MAX_SEQ_LENGTH, 
                            doc_stride=DOC_STRIDE, 
                            max_query_length=MAX_QUERY_LENGTH, 
                            overwrite_cache=True):
    
    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, use_fast=False)
    # Load data features from cache or dataset file
    input_dir = data_dir if data_dir else "."
    cached_features_file = os.path.join(
        input_dir,
        "cached_{}_{}_{}".format(
            "dev",
            list(filter(None, model_name_or_path.split("/"))).pop(),
            str(max_seq_length),
        ),
    )

    # Init features and dataset from cache if it exists
    if os.path.exists(cached_features_file) and not overwrite_cache:
        logger.info("Loading features from cached file %s", cached_features_file)
        features_and_dataset = torch.load(cached_features_file)
        features, dataset, examples = (
            features_and_dataset["features"],
            features_and_dataset["dataset"],
            features_and_dataset["examples"],
        )
    else:

        processor = SquadV2Processor()

        examples = processor.get_dev_examples(data_dir, filename=predict_file)

        features, dataset = squad_convert_examples_to_features(
            examples=examples,
            tokenizer=tokenizer,
            max_seq_length=max_seq_length,
            doc_stride=doc_stride,
            max_query_length=max_query_length,
            is_training=False,
            return_dataset="pt",
            threads=1,
        )


    return dataset, examples, features

In [10]:
dataset, examples, features = load_and_cache_examples(BERT_MODEL_HF_PATH)

add example index and unique id: 100%|██████████| 11873/11873 [00:00<00:00, 732402.44it/s]


In [11]:
print(f'There are {len(examples)} in the dev dataset')

There are 11873 in the dev dataset


The list of examples contains objects of type **transformers.data.processors.squad.SquadExample**. We use the function below to extract the information we want from such objects. More specifically: **'qid'**, **'question_text'**, **'context_text'** and **'answer'**

We will firstly create some extra variables to help on manipulation of data

In [12]:
# generate some maps to help us identify examples of interest
qid_to_example_index = {example.qas_id: i for i, example in enumerate(examples)}
qid_to_has_answer = {example.qas_id: bool(example.answers) for example in examples}
answer_qids = [qas_id for qas_id, has_answer in qid_to_has_answer.items() if has_answer]
no_answer_qids = [qas_id for qas_id, has_answer in qid_to_has_answer.items() if not has_answer]

And also, the function below to help on extracting information given a `qid` (question unique identifier)

In [13]:
def display_example(qid:str):    
    from pprint import pprint

    idx = qid_to_example_index[qid]
    q = examples[idx].question_text
    c = examples[idx].context_text
    a = [answer['text'] for answer in examples[idx].answers]
    
    print(f'Example {idx} of {len(examples)}\n---------------------')
    print(f"Q: {q}\n")
    print("Context:")
    pprint(c)
    print(f"\nTrue Answers:\n{a}")

### Positive Example

In [14]:
display_example(answer_qids[1300])

Example 2548 of 11873
---------------------
Q: Where on Earth is free oxygen found?

Context:
("Free oxygen also occurs in solution in the world's water bodies. The "
 'increased solubility of O\n'
 '2 at lower temperatures (see Physical properties) has important implications '
 'for ocean life, as polar oceans support a much higher density of life due to '
 'their higher oxygen content. Water polluted with plant nutrients such as '
 'nitrates or phosphates may stimulate growth of algae by a process called '
 'eutrophication and the decay of these organisms and other biomaterials may '
 'reduce amounts of O\n'
 '2 in eutrophic water bodies. Scientists assess this aspect of water quality '
 "by measuring the water's biochemical oxygen demand, or the amount of O\n"
 '2 needed to restore it to a normal concentration.')

True Answers:
['water', "in solution in the world's water bodies", "the world's water bodies"]


### Negative Example

The other 50% of questions in the test set do not have an answer. This is important as in a real life Q&A system, our model needs to learn when **to not answer**.

In [15]:
display_example(no_answer_qids[1254])

Example 2564 of 11873
---------------------
Q: What happened 3.7-2 billion years ago?

Context:
("Free oxygen gas was almost nonexistent in Earth's atmosphere before "
 'photosynthetic archaea and bacteria evolved, probably about 3.5 billion '
 'years ago. Free oxygen first appeared in significant quantities during the '
 'Paleoproterozoic eon (between 3.0 and 2.3 billion years ago). For the first '
 'billion years, any free oxygen produced by these organisms combined with '
 'dissolved iron in the oceans to form banded iron formations. When such '
 'oxygen sinks became saturated, free oxygen began to outgas from the oceans '
 '3–2.7 billion years ago, reaching 10% of its present level around 1.7 '
 'billion years ago.')

True Answers:
[]


# Metrics for Q&A Systems

When measuring the performance of a machine learning system, we need to think about both **model** and **customer** metrics.

Q&A systems are usually measured by two dominant metrics: **F1** and **Exact Match (EM)**. They are computed on individual **Question & Answer** pairs. When multiple correct answers are available for a given question, the maximum score over all possible correct answers is computed. Overall **EM** and **F1** scores are computed for a model by averaging over the individual example scores


## Exact Match

For each Q&A Pair, if the **characters** of the model's prediction are an exact match of the characters of any of the True Answer(s), **EM=1**, otherwise **EM=0**. This is a strict all-or-nothing metric, which may have little value for final customers of a **Q&A System**. It may be beneficial only when assessing against a negative example; if the model predicts any text at all, it automatically receives a **0** for that example

## F1 Score

Almost all classificattion problems rely on F1 score to measure model performance. It is mostly appropriate when we care equally about precision and recall. On a **Q&A system**, however, it is computed over the individual words in the prediction against those in the **True Answer**. The number of shared words between the prediction and the trust is the basis of the F1 score. While **precision** is the ration between the number of shared words to the total number of words in the prediction; Recall is the ratio of the number of shared words to the total number of words in the ground truth

## Latency 

Lateny is an important metric for ML Systems. In the Q&A example its of the utmost importance when the system is used in a conversational application. For instance: Alexa and Google home are devices that have very strict latency constraints as the uses expects an answer with a few seconds after the question was asked. When updating models we should take this dimension according to the application of the system

## Answer Rate

In Q&A Systems, models that attempt to answer every question are often perceived as innacurate. The system should only provide output when confident enough to do so, in other words, when the probabilities of prodictions are above certain threshold. In some applications, a model should also be able to say "I don't know" or "The context has not enough information to answer the question".

# Q&A Models

Question and Answering makes use of Large Language Models (LLMs) as any other classification problem in NLP. The main difference relies on how the input and output is provided to the model. Generally speaking models are trained to match the **true answer** to the **question** as they are provided together as an input to the model.

## BERT

BERT, or Bidirectional Encoder Representations from Transformers, is a neural approach to pre-train language representations which obtains near state-of-the-art results on a wide array of Natural Language Processing (NLP) tasks, including SQuAD Question Answering dataset

Developed in 2019 BERT achieves **80.422** in the **EM** score and **83.118** in the **F1** score

BERT-base has **110 million** parameters and BERT-large has **340 million** parameters

## Model Parameters Comparison

As Large Language Models were developed, the amount of parameters in these models have grown exponentially. Although this improves model performance it comes at a cost: **Latency**. As we will discuss, for use cases where inference is done on batches that may have less impact, however, on real time systems such as voice assistants or web search, latency plays a major role on deciding whether one model is better than the other.

### BERT Input

[CLS] context [SEP] question [SEP] [PAD] [PAD] [PAD]

**context** = "The Intergovernmental Panel on Climate Change (IPCC) is a scientific intergovernmental body under the auspices of the United Nations."

**question** = "What organization is the IPCC a part of?"

**after being merged by the tokenizer**:
```
"[CLS] The Intergovernmental Panel on Climate Change (IPCC) is a scientific intergovernmental body under the auspices of the United Nations. [SEP] What organization is the IPCC a part of? [SEP] [PAD] [PAD] [PAD]"
```


**token-id format**:

[101, 1109, 11300, 2758, 24472, 15595, 20339, 1113, 13540, 9091, 113, 14274, 12096, 114, 1110, 170, 3812,
 9455, 2758, 24472, 15595, 1404, 1223, 1103, 22105, 1104, 1103, 1244, 3854, 119, 102, 1327, 2369, 1110, 1103,
 14274, 12096, 170, 1226, 1104, 136, 102, 0, 0, 0]

## Loading Pre Trained BERT from Huggingface repository

In [16]:
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_HF_PATH, use_fast=False)
model = AutoModelForQuestionAnswering.from_pretrained(BERT_MODEL_HF_PATH)

Some weights of the model checkpoint at twmkn9/bert-base-uncased-squad2 were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


## Utility Functions

Given a **Question ID**, **Model** and **Tokenizer** we get an answer text. In here we get the maximum probability of beginning and end for the answer in the Softmax output

In [17]:
def get_prediction(qid: str, model:AutoModelForQuestionAnswering, tokenizer:AutoTokenizer):
    # given a question id (qas_id or qid), load the example, get the model outputs and generate an answer
    question = examples[qid_to_example_index[qid]].question_text
    context = examples[qid_to_example_index[qid]].context_text

    inputs = tokenizer.encode_plus(question, context, return_tensors='pt')

    outputs = model(**inputs)
    answer_start = torch.argmax(outputs[0])  # get the most likely beginning of answer with the argmax of the score
    answer_end = torch.argmax(outputs[1]) + 1 

    answer = tokenizer.convert_tokens_to_string(tokenizer.convert_ids_to_tokens(inputs['input_ids'][0][answer_start:answer_end]))

    return answer

We create a simple function that given an **example** list it extracts the gold answers

In [18]:
def get_gold_answers(example):
    """helper function that retrieves all possible true answers from a squad2.0 example"""
    
    gold_answers = [answer["text"] for answer in example.answers if answer["text"]]

    # if gold_answers doesn't exist it's because this is a negative example - 
    # the only correct answer is an empty string
    if not gold_answers:
        gold_answers = [""]
        
    return gold_answers

For Metrics like **Exact Match** we need to make sure that texts are normalized so we can compare on a character level

In [19]:
# these functions are heavily influenced by the HF squad_metrics.py script
def normalize_text(s: str):
    """Removing articles and punctuation, and standardizing whitespace are all typical text processing steps."""
    import string, re

    def remove_articles(text):
        regex = re.compile(r"\b(a|an|the)\b", re.UNICODE)
        return re.sub(regex, " ", text)

    def white_space_fix(text):
        return " ".join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))
